<font size="6"><b>Module 7 – Dimensionality Reduction</b></font>

<img src="images_module07/module07.png" width="600"/>

Many Machine Learning problems involve thousands or even millions of features for each training instance. Not only do all these features make training extremely slow, but they can also make it much harder to find a good solution, as we will see. **This problem is often referred to as the curse of dimensionality.** 

In real-world problems, it is often possible to reduce the number of features considerably, turning an intractable problem into a tractable one.
- Consider the MNIST images: 
    - drop border pixels
    - combine highly correlated neighbouring pixels 

**NOTE:** Reducing dimensionality does cause some information loss (just like compressing an image to JPEG can degrade its quality), so even though it will speed up training, it may make your system perform slightly worse. 
- It also makes your pipelines a bit more complex and thus harder to maintain. 
- If training isn't already too slow, you should first try to train your system with the original data before considering using dimensionality reduction. 

**In some cases, reducing the dimensionality of the training data may filter out some noise and unnecessary details and thus result in higher performance, but in general it won’t; it will just speed up training.**


Here's what we will cover in this module:
- We will discuss **the curse of dimensionality and get a sense of what goes on in high-dimensional space.** 
- We will consider the two main approaches to dimensionality reduction (projection and Manifold Learning), and we will go through three of the most popular dimensionality reduction techniques: 
    - PCA, 
    - Kernel PCA
    - and LLE. 
    
    
Note, we will be using plotly's 3D viz tools in this module, so you may need to install both plotly and the jupyterlab plotly extension:
```
    pip install plotly
    conda install -c conda-forge jupyterlab-plotly-extension -y 
```

You may also need to restart/rebuild jupyter after installation. One this is done, run the code below to make sure everything is working:


In [6]:
%pip install "nbformat>=4.2.0"

  Using cached nbformat-5.11.1-py3-none-any.whl.metadata (3.7 kB)
Using cached nbformat-5.11.1-py3-none-any.whl (79 kB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# run this code to test your plotly 3D viz extension in jupyter
import plotly.graph_objects as go
import numpy as np
from math import isclose

# Helix equation
t = np.linspace(0, 10, 50)
x, y, z = np.cos(t), np.sin(t), t

fig = go.Figure(data=[go.Scatter3d(x=x, y=y, z=z,
                                   mode='markers')])
fig.show()

**SETUP**

In [1]:
# Common imports
import numpy as np
import pandas as pd
import os

# Plotly imports
import plotly.graph_objects as go
import numpy as np

# to make this notebook's output stable across runs
np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

ModuleNotFoundError: No module named 'numpy'

##### The Curse of Dimensionality 

We are so used to living in three dimensions that our intuition fails us when we try to imagine a high-dimensional space. 

Even a **basic 4D hypercube is incredibly hard to picture in our minds:**

<img src="images_module07/01_4d.png" width="600"/>


It turns out that many things behave very differently in high-dimensional space. For example:
- if you pick a random point in a unit square (a 1 × 1 square), it will have only about a 0.4% chance of being located less than 0.001 from a border (in other words, it is very unlikely that a random point will be “extreme” along any dimension). 

**In a 10,000-dimensional unit hypercube, this probability is greater than 99.999999%.**
- Most points in a high-dimensional hypercube are very close to the border.

Here is a more troublesome difference: 
- If you pick two points randomly in a unit square, the distance between these two points will be, on average, roughly 0.52. 
- If you pick two random points in a unit 3D cube, the average distance will be roughly 0.66. 
- In a 1,000,000-dimensional hypercube? The average distance, believe it or not, will be about 408.25 (roughly $\sqrt{1000000/6}$) 



In [ ]:
# let's come up with an equation that computes the average distance
# of two randomly-selected points in a hypercube of dimension 'd'
def avg_distance(d, iters=1000):
    distances = [] 
    for i in range(iters):
        a = np.array([np.random.rand() for di in range(d)])
        b = np.array([np.random.rand() for di in range(d)])
        distances.append(np.linalg.norm(a-b))
    return np.mean(distances)

avg_distance(2)

In [ ]:
avg_distance(3)

In [ ]:
avg_distance(310)

In [ ]:
# showing distances for dimensions of 1 through 10
distances = []
for i in range(100):
    distances.append((i, avg_distance(i)))

distances[0:10]

In [ ]:
# plot depicting this increase
i_s, i_distances = list(zip(*distances))
plt.plot(i_s, i_distances)
plt.show()

How can two points be so far apart when they both lie within the same unit hypercube? 
- There’s just plenty of space in high dimensions. 
- As a result, high-dimensional datasets are at risk of being very sparse: 
    - most training instances are likely to be far away from each other. 
    - This also means that a new instance will likely be far away from any training instance, making predictions much less reliable than in lower dimensions, since they will be based on much larger extrapolations. 
    
In short, the more dimensions the training set has, **the greater the risk of overfitting it**. 

In theory, one solution to the curse of dimensionality could be to **increase the size of the training set to reach a sufficient density of training instances**. 
- Unfortunately, in practice, the number of training instances required to reach a given density grows exponentially with the number of dimensions. 
- With just 100 features (significantly fewer than in the MNIST problem), you would need **more training instances than atoms in the observable universe in order for training instances to be within 0.1 of each other on average,** assuming they were spread out uniformly across all dimensions. 



Before we dive into specific dimensionality reduction algorithms, let’s take a look at the two main approaches to reducing dimensionality: 
- projection
- Manifold Learning. 


# Main Approaches for Dimensionality Reduction

## Projection methods

In most real-world problems, **training instances are not spread out uniformly across all dimensions.** 
- Many features are **almost constant**
- Others are **highly correlated**

As a result, **all training instances lie within (or close to) a much lower-dimensional subspace of the high-dimensional space.** 

Take a look at the 3D data in the plot below. Notice that all training instances lie close to a plane: 
- this is a **lower-dimensional (2D) subspace of the high-dimensional (3D) space.**

<img src="images_module07/02_2dhyperplane.png" width="500"/>

If we project every training instance **perpendicularly** onto this subspace (as shown below), we get the new 2D dataset.

<img src="images_module07/03_2dhyperplane.png" width="500"/>



The projection of vector $\vec{x}$ onto a line $\vec{v}$ is given by this formula:

$\textbf{proj}_{\textbf{V}}{\vec{x}} = \dfrac{\vec{x} \cdot \vec{v}}{\left \Vert \vec{v} \right \| ^2} \times \vec{v}$

If $\vec{v}$ is a unit vector (its length is 1), the above equation simplifies to :

$\textbf{proj}_{\textbf{V}}{\vec{x}} = (\vec{x} \cdot \vec{v}) \times \vec{v}$



This naturally extends to subspaces that span multiple dimensions. If the basis vectors for the subspace form an orthonormal set, one way to do this is to just project the vector $\vec{x}$ onto each of said basis vectors and add the projections together. A more general and efficient approach is to use matrix operations: 

$\textbf{proj}_{\textbf{V}}{\vec{x}} = \matrix{A}(\matrix{A}^\top \matrix{A})^{-1} \matrix{A}^\top \vec{x}$

Where 
- $\matrix{A}$ is a matrix composed of all the basis vectors of subspace V (linearly independent vectors that span V)

Note that when projecting onto a line, $\matrix{A}^\top \matrix{A}$ was just a number; now it is a square matrix. So instead of dividing by $\vec{a}^\top \vec{a}$ we now have to divide by $\matrix{A}^\top \matrix{A}$.

Similar to the case above where we are projecting onto a line, if all the vectors in $\matrix{A}$ are orthogonal to one another, and they each have a length of 1 (and thus, they form an orthonormal basis for V), then $(\matrix{A}^\top \matrix{A})^{-1}$ is simply equal to the identity matrix. We can once again simplify this equation as follows:

$\textbf{proj}_{\textbf{V}}{\vec{x}} = \matrix{A}\matrix{A}^\top \vec{x}$

Let's test this out by applying these equations and projecting some data $X$ onto a subspace:


In [ ]:
# dataset; creates a circle in (X, y), but distributes the
# datapoints randomly along the Z axis, to give us a 3D distribution
from sklearn import datasets
n=100
X,y = datasets.make_circles(n_samples=n, noise=None)
X = np.hstack([X,np.reshape(np.random.rand(n),(n,1))]).T

# orthonormal Subspace in R^3 V=span(v1, v2)
# (these define the axes of a 2D plane, embedded in 3D space)
v1 = np.array([1/3,2/3,2/3])
v2 = np.array([2/3,1/3,-2/3])
A = np.stack([v1,v2]).T

# this simply computes evenly-spaced points along this 2D plane
# so we can better visualize it later
vectors = []
k = 1
for i in np.linspace(-k,k,50):
    for j in np.linspace(-k,k,50):
        vectors.append((A@np.array([i,j])))
subspace = np.stack(vectors)

# we prove the orthonormality of our basis vectors
v1_norm = sum(v1*v1)
v2_norm = sum(v2*v2)
v1_v2 = np.dot(v1, v2)
print("A rank:", np.linalg.matrix_rank(A))
print('V1 norm:', v1_norm)
print('V2 norm:', v2_norm)
print('V1 dot V2:', v1_v2)
if v1_norm == 1 and v2_norm == 1 and isclose(v1_v2, 0, abs_tol=1e-9):
    print("Basis vectors are orthonormal")
else:
    print("Basis vectors are not orthonormal")


In [ ]:
# === Projection of Dataset onto new orthonormal basis ===
# Method 1: project x onto each basis vector and sum (only works with orthonormal bases)
# x=v+w where v is member of subspace V in R3 and w is member of orthogonal complement to subspace V in R3
projected_vectors = []
for xi in X.T:
    v1_proj = xi.dot(v1.T)/(v1.dot(v1.T)) * v1
    v2_proj = xi.dot(v2.T)/(v2.dot(v2.T)) * v2
    projected_vectors.append(v1_proj+v2_proj)
proj1 = np.stack(projected_vectors)
print(proj1[0:5])

# Method 2: Matrix projection
proj2 = (A@np.linalg.inv(A.T@A)@A.T@X).T 
print(proj2[0:5])

# Method 3: Matrix projection assuming orthonormal basis
proj3 = (A@A.T@X).T
print(proj3[0:5])

# All projections should be equal
assert np.allclose(proj1, proj2)
assert np.allclose(proj1, proj3)
assert np.allclose(proj2, proj3)

In [ ]:
subspace.shape

In [ ]:
# === Visualization of projection ===
# projection onto subspace defined by 'A'; will work for both
# orthonormal and non-orthonormal vectors
if v1_norm == 1 and v2_norm == 1 and isclose(v1_v2, 0, abs_tol=1e-9):
    print("Basis vectors are orthonormal")
    proj = (A@A.T@X).T
else:
    print("Basis vectors are NOT orthonormal")
    proj = (A@(np.linalg.inv(A.T@A))@A.T@X).T


# Plot data, subspace, and projection
fig = go.Figure(data=[go.Scatter3d(mode='markers', x=X[0,:],    y=X[1,:],    z=X[2,:], opacity=1.0, name='data'),
                      go.Scatter3d(mode='markers', x=subspace[:,0], y=subspace[:,1], z=subspace[:,2], opacity=0.4, name='subspace_V'),
                      go.Scatter3d(mode='markers', x=proj[:,0], y=proj[:,1], z=proj[:,2], opacity=0.6, name='projV_data')
                     ])
fig.update_traces(marker=dict(size=2))

# Normal vector
normal_vector = np.cross(v1, v2) *0.5
normal_line = np.stack([normal_vector,normal_vector*-1])
colors = [[0, 'red']]
line_marker = dict(color='red', width=4)
fig.add_trace(go.Cone(x=[*normal_line[:,0]], y=[*normal_line[:,1]], z=[*normal_line[:,2]], u=[*normal_line[:,0]], v=[*normal_line[:,1]], w=[*normal_line[:,2]], sizeref=0.20, showscale=False, opacity=1, colorscale=colors))
fig.add_trace(go.Scatter3d(x=[*normal_line[:,0]], y=[*normal_line[:,1]], z=[*normal_line[:,2]], mode='lines', opacity=1, line=line_marker, name='normal_v'))
fig.show()

In [ ]:
# let's re-compute this, but using a different orthonormal
# subspace for projection, that does a better of job of capturing
# the underlying trends in the data

v1 = np.array([1,0,0])
v2 = np.array([0,1,0])
A = np.stack([v1,v2]).T

vectors = []
k = 1
for i in np.linspace(-k,k,50):
    for j in np.linspace(-k,k,50):
        vectors.append((A@np.array([i,j])))
subspace = np.stack(vectors)

v1_norm = sum(v1*v1)
v2_norm = sum(v2*v2)
v1_v2 = np.dot(v1, v2)


if v1_norm == 1 and v2_norm == 1 and v1_v2 == 0:
    print("Basis vectors are orthonormal")
    proj = (A@A.T@X).T
#     assert np.allclose(proj, (A@np.linalg.inv(A.T@A)@A.T@X).T)
else:
    print("Basis vectors are NOT orthonormal")
    proj = (A@(np.linalg.inv(A.T@A))@A.T@X).T


# Plot data, subspace, and projection
fig = go.Figure(data=[go.Scatter3d(mode='markers', x=X[0,:],    y=X[1,:],    z=X[2,:], opacity=1.0, name='data'),
                      go.Scatter3d(mode='markers', x=subspace[:,0], y=subspace[:,1], z=subspace[:,2], opacity=0.4, name='subspace_V'),
                      go.Scatter3d(mode='markers', x=proj[:,0], y=proj[:,1], z=proj[:,2], opacity=0.6, name='projV_data')
                     ])
fig.update_traces(marker=dict(size=2))

# Normal vector
normal_vector = np.cross(v1, v2) *0.5
normal_line = np.stack([normal_vector,normal_vector*-1])
colors = [[0, 'red']]
line_marker = dict(color='red', width=4)
fig.add_trace(go.Cone(x=[*normal_line[:,0]], y=[*normal_line[:,1]], z=[*normal_line[:,2]], u=[*normal_line[:,0]], v=[*normal_line[:,1]], w=[*normal_line[:,2]], sizeref=0.20, showscale=False, opacity=1, colorscale=colors))
fig.add_trace(go.Scatter3d(x=[*normal_line[:,0]], y=[*normal_line[:,1]], z=[*normal_line[:,2]], mode='lines', opacity=1, line=line_marker, name='normal_v'))
fig.show()

However simple projection is not always the best approach. Take the case of the **swiss roll**:

<img src="images_module07/04_swissroll.png" width="500"/>

Simple projecting onto a 2D hyperplane would squash different layers of the swiss role together (as seen in the left image below). 

What you really want to do is **unroll** the data (as see on the right):

<img src="images_module07/05_swissroll.png" width="500"/>


## Manifold Learning

The Swiss roll is an example of a **2D manifold**:
- a 2D manifold is a 2D shape that can be bent and twisted in a higher-dimensional space. 

More generally: 
- a $d$ dimensional manifold is a part of an $n$ dimensional space (where $d < n$ ) that locally resembles a $d$ dimensional hyperplane. 
- In the case of the Swiss roll, d = 2 and n = 3: 
    - it locally resembles a 2D plane, but it is rolled in the third dimension. 
    
Many dimensionality reduction algorithms work by modeling the manifold on which the training instances lie; this is called Manifold Learning. It relies on the manifold assumption , also called the **manifold hypothesis**:
- which holds that most **real-world high-dimensional datasets lie close to a much lower-dimensional manifold**. 
- This assumption is very often empirically observed. 
- Think about the MNIST dataset: all handwritten digit images have some similarities. 

The manifold assumption is often accompanied by another implicit assumption: 
- that the task at hand (e.g., classification or regression) will be simpler if expressed in the lower-dimensional space of the manifold.

However, this implicit assumption does not always hold.
- In the bottom row of the figure below, the decision boundary is located at $x_1 = 5$. 
- This decision boundary looks very simple in the original 3D space (a vertical plane), but it looks **more complex in the unrolled manifold (a collection of four independent line segments)**. 

In short, reducing the dimensionality of your training set before training a model will **usually speed up training, but it may not always lead to a better or simpler solution;** it all depends on the dataset. 

<img src="images_module07/06_swissroll.png" width="500"/>

# Principal Component Analysis

Principal Component Analysis (PCA) is by far the most popular dimensionality reduction algorithm. The main idea is to **identify the axis (hyperplane) that accounts for the largest amount of variance in the training set, and then project the data onto that hyperplane**:
- It also finds a second axis, orthogonal to the first one, that accounts for the largest amount of remaining variance. 
- In a higher-dimensional dataset, PCA would also find a third axis, orthogonal to both previous axes, and a fourth, a fifth, and so on as many axes as the number of dimensions in the dataset. 
- The $i^{th}$ axis is called the $i^{th}$ principal component (PC) of the data (for each principal component, PCA finds a **zero-centered unit vector pointing in the direction of the PC**) 

Before you can project the training set onto a lower-dimensional hyperplane however, you first need to choose the right hyperplane. This process is illustrated below with a simple 2D dataset:
- On the right is the result of the projection of the dataset onto each of these axes. 
- The projection onto the solid line **preserves the maximum variance**, while the projection onto the dotted line preserves very little variance and the projection onto the dashed line preserves an intermediate amount of variance. 

<img src="images_module07/07_presvar.png" width="500"/>


It seems reasonable to **select the axis that preserves the maximum amount of variance, as it will most likely lose less information than the other projections.** 
- Another way to justify this choice is that it is the axis that **minimizes the mean squared distance between the original dataset and its projection onto that axis.** 
- This is the rather simple idea behind PCA.

So how do we find these principal components? Well, the short answer is a combination of the **Covariance Matrix and Singular Value Decomposition.** But before we dig into these concepts we will first run through a quick linear algebra primer. 

## Linear Algebra Review

### Eigenvalues and Eigenvectors

Let $\mathbf {A}$ be a square $n × n$ matrix. A (non-zero) vector $v$ of dimension $n$ is an eigenvector of $\mathbf {A}$ if it satisfies the linear equation:

${\displaystyle \mathbf {A} \mathbf {v} =\lambda \mathbf {v} }$

where:
- $λ$ is a scalar, termed the **eigenvalue**
- $\mathbf {v}$ is the **eigenvector** corresponding to $λ$

Essentially the eigenvectors are the vectors that the linear transformation  $ \mathbf {A} $ merely **elongates or shrinks**, and the amount that they elongate/shrink by is the **eigenvalue**. The above equation is called the eigenvalue equation or the eigenvalue problem.

This yields an equation for the eigenvalues:

$\mathbf {A} \mathbf {v} =\lambda \mathbf {v}   $

$\\ \mathbf {A} \mathbf {v} - \lambda \mathbf {v} = \mathbf {0}$

$\\ \mathbf {A} \mathbf {v} - \lambda \mathbf{I} \mathbf {v} = \mathbf {0}$

$\\ (\mathbf {A} - \lambda \mathbf{I}) \mathbf {v} = \mathbf {0}$

$\\ p\left(\lambda \right)=\det \left(\mathbf {A} -\lambda \mathbf {I} \right)=0$

We call $p\left(\lambda \right)$ the **characteristic polynomial**. Note that we are looking to find non-zero vectors in the null-space of $\mathbf {A} - \lambda \mathbf{I}$, which means $\mathbf {A} - \lambda \mathbf{I}$ must have a non-trivial null space. If the null-space is non-trivial, we should have a determinant of 0. This is why we why we set the determinant to 0 (finding the roots of the polynomial) to obtain the eigenvalues of A.


Let's take a quick example. Matrix $\mathbf {A}$ is defined as follow:

${\displaystyle \mathbf {A} ={\begin{bmatrix}-3&4&4 \\4&4&-2 \\4&-2&5 \end{bmatrix}}\in \mathbb {R} ^{3\times 3}}$

${\displaystyle \mathbf {A} - \lambda \mathbf{I} ={\begin{bmatrix}-\lambda-3&4&4 \\4&-\lambda+4&-2 \\4&-2&-\lambda+5 \end{bmatrix}}}$

Equivalently we can write it this way:

${\displaystyle \lambda \mathbf{I} - \mathbf {A}={\begin{bmatrix}\lambda+3&-4&-4 \\-4&\lambda-4&2 \\-4&2&\lambda-5 \end{bmatrix}}}$

We then calculate $\det \left(\lambda \mathbf {I} - \mathbf {A} \right)$ using your method of choice (e.g. the rule of saurus), and we get the **characteristic polynomial**:

$p\left(\lambda \right) = l^3-6l^2-43l+256$

For which it can be shown that the roots are:

$l\approx \:5.78915\dots ,\:l\approx \:-6.54527\dots ,\:l\approx \:6.75611\dots $

From here, we can solve for the corresponding eigenvectors (or eigenspaces) by simply plugging in the eigenvalues (one at a time) into the system below (put into reduced row echelon form to solve for values of $\mathbf{v}$):

${\begin{bmatrix}\lambda_1+3&-4&-4 \\-4&\lambda_2-4&2 \\-4&2&\lambda_3-5 \end{bmatrix}} \mathbf{v_1}= \mathbf{0}$

You are encouraged to test this out by hand yourself. That said,if you have a solid understanding of the theory already, python makes it much easier to find eigenvalues and eigenvectors:

In [ ]:
A = np.array([[-3,  4,  4],
              [ 4,  4, -2],
              [ 4, -2,  5]])

eigen_vals, eigen_vecs = np.linalg.eig(A)
eigen_vals

In [ ]:
i=0
for i in range(3):
    eigen_I = np.identity(3)*eigen_vals[i]
    # Should produce 0 vector
    print(np.round((eigen_I - A)@eigen_vecs[:,i], 5))

### Eigendecomposition

Let $\mathbf {A}$ be a square $n × n$ matrix with has $n$ linearly independent eigenvectors $q_i$ (where $i = 1, ..., n$). Then $\mathbf {A}$ can be factorized as:

${\displaystyle \mathbf {A} =\mathbf {Q} \mathbf {\Lambda } \mathbf {Q} ^{-1}}$

where:
- $\mathbf {Q}$ is the square $n × n$ matrix whose $i^{th}$ column is the eigenvector $q_i$ of $ \mathbf {A}$
- $\mathbf \Lambda$ is the **diagonal matrix** whose diagonal elements are the corresponding eigenvalues, $\Lambda_{ii} = λi$. 

> In other words, all non-singular (i.e. diagonalizable) matrix transformations can be decomposed into a pure rotation, followed by a pure re-scaling, followed by another pure rotation. Remember that a matrix acting on a vector simply maps that vector to another one within the same space. 

The $n$ eigenvectors $q_i$ are usually normalized, but they need not be. A non-normalized set of $n$ eigenvectors, $v_i$ can also be used as the columns of $\mathbf {Q}$. That can be understood by noting that the magnitude of the eigenvectors in $\mathbf {Q}$ gets canceled in the decomposition by the presence of $\mathbf {Q} ^{-1}$. 

The decomposition can be derived from the fundamental property of eigenvectors:

${\displaystyle {\begin{aligned}\mathbf {A} \mathbf {v} &=\lambda \mathbf {v} \\ \mathbf {A} \mathbf {Q} &=\mathbf {Q} \mathbf {\Lambda } \\ \mathbf {A} &=\mathbf {Q} \mathbf {\Lambda } \mathbf {Q} ^{-1}\end{aligned}}}$

Again, its important to note that only **diagonalizable matrices** can be factorized in this way. This basically means we need $n$ linearly independent eigenvectors or $n$ distinct eigenvalues of $\mathbf {A}$ (if this isn't satisfied, we would not be able to find $\mathbf {Q} ^{-1}$). 

Recall that this is incredibly useful for finding the $n^{th}$ power of a matrix:

$\mathbf {A^n} =\mathbf {Q} \mathbf {\Lambda^n } \mathbf {Q} ^{-1}$






In [ ]:
A = np.array([[-3.,  4.,  4.],
              [ 4.,  4., -2.],
              [ 4., -2.,  5.]])

print("A rank:", np.linalg.matrix_rank(A))

In [ ]:
# Get eigenvectors
l, Q =  np.linalg.eig(A)
print("Q rank:", np.linalg.matrix_rank(Q))

In [ ]:
# Create diagonal matrix with eigenvalues
L = np.diag(l)

# Get Q inverse
Q_inv = np.linalg.inv(Q)

# Decomp
A_decomp = np.round(Q@L@Q_inv,6)

A == A_decomp

In [ ]:
# Find the nth power of a matrix
n = 10
out = A
for _ in range(n-1):
    out = out@A
A_decomp = np.round(Q@L**n@Q_inv,5)
out == A_decomp

### Singular Value Decomposition



Alright, now we can dive into the aforementioned topic at hand, the Singular Value Decomposition (SVD).

SVD is the generalization of the eigendecomposition to non-square matrices, e.g. those with ($m \times n$ ) dimensions. Unlike the eigendecomposition which **only applies to diagonalizeable square matrices**, SVD can be used to decompose ANY non-zero matrix:

Recall that the SVD decomposes a matrix into 3 separate matrices as follows:

$\mathbf{A} = \mathbf{U}\mathbf{\Sigma} \mathbf{V^{T}}$

where:
- $\mathbf{U}$ is a rotation matrix (an $m \times m$ orthogonal matrix)
- $\mathbf{\Sigma}$ is a scaling & stretching matrix (an $m \times n$ diagonal matrix)
- $\mathbf{V}$ is a rotation matrix (an $n \times n$ orthogonal matrix)

But how do we get these 3 separate matrices?

Let's start by taking $\mathbf{A^T}$ and applying it to both side of the equation. Note that the matrix $AAᵀ$ and $AᵀA$ are very special in linear algebra. Consider any $m × n$ matrix $\mathbf{A}$, we can multiply it with $Aᵀ$ to form $AAᵀ$ and $AᵀA$ separately. These matrices are:
- symmetrical,
- square,
- at least positive semidefinite (eigenvalues are zero or positive),
- both matrices have the same positive eigenvalues, and
- both have the same rank $r$ as $\mathbf{A}$

Furthermore, with symmetric matrices we can choose for their eigenvectors to be orthonormal (perpendicular to each other with unit length) — this is a fundamental property for symmetric matrices, and essential for the SVD:

$\mathbf{A^T}\mathbf{A} = (\mathbf{V}\mathbf{\Sigma^{T}} \mathbf{U^{T}})  \mathbf{U}\mathbf{\Sigma} \mathbf{V^{T}}$

Since $\mathbf{U}$ is an orthogonal matrix $\mathbf{U^T} = \mathbf{U^{-1}}$, so we can simplify:

$\\ \mathbf{A^T}\mathbf{A} = \mathbf{V}\mathbf{\Sigma^{T}} \mathbf{\Sigma} \mathbf{V^{T}}$

$\\ \mathbf{A^T}\mathbf{A} = \mathbf{V}\mathbf{\Sigma^{2}} \mathbf{V^{T}}$

$\\ \mathbf{A^T}\mathbf{A} = \mathbf{V}\mathbf{\Sigma^{2}} \mathbf{V^{-1}}$





Now we have the **eigendecomposition** form (except we are simply squaring the eigenvalues). It is important to note that $\mathbf{A}$ is any old rectangular matrix, but $\mathbf{A^T}\mathbf{A}$ is a symmetric matrix, which has $n$ perpendicular eigenvectors and $n$ real eigenvalues (this makes this whole process possible). 

We can derive $\mathbf{U}$ in a similar fashion:

$\\ \mathbf{A}\mathbf{A^T} = \mathbf{U}\mathbf{\Sigma^{T}} \mathbf{\Sigma} \mathbf{U^{T}}$

$\\ \mathbf{A}\mathbf{A^T} = \mathbf{U}\mathbf{\Sigma^{2}} \mathbf{U^{T}}$

$\\ \mathbf{A}\mathbf{A^T} = \mathbf{U}\mathbf{\Sigma^{2}} \mathbf{U^{-1}}$

Alternatively, once we have $\mathbf{V}$ we could use it to solve for $\mathbf{U}$ (keeping in mind that columns of $\mathbf{U}$ should be unit vectors, as it is an orthonormal matrix):

$\mathbf{A}\mathbf{V} = \mathbf{U}\mathbf{\Sigma}$

So we can think of our main 3 SVD matrices as follows:
- $\mathbf{U}$ are the $n$ eigenvectors of $\mathbf{AA^T}$: the **left singular vectors**
- $\mathbf{V}$ are the $n$ eigenvectors of $\mathbf{A^T A}$: the **right singular vectors**
- $\mathbf{\Sigma}$ are the square roots of the shared $n$ eigenvalues ($\lambda^{1/2})$ for $\mathbf{AA^T}$ and $\mathbf{A^T A}$: the **the singular values**

And again, we know that both $\mathbf{AA^T}$ and $\mathbf{A^T A}$ will have $n$ perpendicular eigenvectors and $n$ real eigenvalues because they are both symmetric matrices. This is why we know that both $\mathbf{U}$ and $\mathbf{V}$ will be orthogonal matrices. 


Let's test this out in numpy:

In [ ]:
A = np.array([[-3.,  4.,  4.],
              [ 4.,  4., -2.],
              [ 4., -2.,  5.]])

print("A rank:", np.linalg.matrix_rank(A))

In [ ]:
eigen_vals1, V = np.linalg.eig(A.T@A)
eigen_vals2, U = np.linalg.eig(A@A.T)

Eigenvalues should be the same because eigenvalues of AB == eigenvalues of BA

In [ ]:
assert np.allclose(eigen_vals1,eigen_vals2)

Note: we actually use 𝐀𝐕=𝐔𝚺 to calculate U, as computing eigenvalues of $\mathbf{A^TA}$ and $\mathbf{AA^T}$ independently can cause sign ambiguity
https://math.stackexchange.com/questions/2359992/how-to-resolve-the-sign-issue-in-a-svd-problem

In [ ]:
S = np.diag(eigen_vals1)**(1/2)
U = A@V@np.linalg.inv(S)

print(U.shape)
print(S.shape)
print(V.T.shape)

U@S@V.T

In [ ]:
# Using built in numpy SVD
U_, S_, V_ = np.linalg.svd(A)
U_@np.diag(S_)@V_

In [ ]:
# Manual U
U

In [ ]:
# numpy svd U
U_

You may have noticed that the `np.linalg.svd` implementation returns the eigenvectors in a different order. This is because the SVD algorithm sorts the eigenvalues from highest to lowest, and arranged the eigenvectors correspondingly.  We will explain the reasoning for this shortly. We can produce the same result by simply sorting our eigenvectors in the same way:



In [ ]:
eigen_order = np.argsort(-eigen_vals1)
V_sorted = V[:,eigen_order]
S_sorted = np.diag(eigen_vals1[eigen_order])**(1/2)
U_sorted = A@V_sorted@np.linalg.inv(S_sorted)

# Sorted U should match U_
U_sorted

In [ ]:
# Note both -v and v are in null space of (I𝜆-A)
# This can cause "sign ambiguity"
for i in range(3):
    # Print negative
    print(np.round(((np.identity(3)*eigen_vals1[eigen_order][i]) -(A@A.T)) @ U[:,i],5))
    # Print positive
    print(np.round(((np.identity(3)*eigen_vals1[eigen_order][i]) -(A@A.T)) @ -U[:,i],5))


### SVD Intuition

SVD can be difficult to interpret, but if you look past the math and focus on the core concepts at play, it is actually quite simple. The main principle you need to understand is that **any vector can be expressed in terms of:**
1. Projection directions represented by orthogonal  unit vectors  (v₁, v₂, …)
2. The lengths of the projected vector onto said unit vectors (sₐ₁, sₐ₂, …), which tell us how much of the vector is contained in each direction

For instance, vector $\mathbf{a}$  in the graphic below can be represented as a combination of the unit vectors v1 and v2, as well as the lengths of its projections onto those unit vectors (s1 and s2).

![](images_module07/projection.png)


So if we take the dot product of any vector with a matrix containing its orthogonal unit vectors, we get the projections lengths $\mathbf{S}$:


$\mathbf{a^T} \cdot \mathbf{V} =  {\begin{bmatrix}a_{x}&a_{y}\end{bmatrix}} {\begin{bmatrix}v_{1x}&v_{2x} \\ v_{1y}&v_{2y} \end{bmatrix}} = {\begin{bmatrix}s_{a1}&s_{a2}\end{bmatrix}} = \mathbf{S}$


And we can generalize this to any number of points:


$\mathbf{A} \cdot \mathbf{V} =  {\begin{bmatrix}a_{x}&a_{y}&\cdots \\ b_{x}&b_{y}&\cdots \\ \vdots&\vdots&\ddots \end{bmatrix}}
{\begin{bmatrix}v_{1x}&v_{2x}&\cdots \\ v_{1y}&v_{2y}&\cdots \\ \vdots&\vdots&\ddots \end{bmatrix}} = {\begin{bmatrix}s_{a1}&s_{a2}&\cdots \\ s_{b1}&s_{b2}&\cdots \\ \vdots&\vdots&\ddots \end{bmatrix}} = \mathbf{S}$

So we have:
$\mathbf{A} \cdot \mathbf{V} = \mathbf{S}$
where:
- $\mathbf{A}$ is a $m \times n$ matrix of our data ($m$=number of data points, $n$=dimension of each point)
- $\mathbf{V}$ is a $n \times n$ matrix composed of the set of orthogonal unit vectors
- $\mathbf{S}$ is a $m \times n$ matrix composed of the lengths of the projections 

Since $\mathbf{V}$ is an orthonormal matrix, we can write:

$\\ \mathbf{A} \cdot \mathbf{V} = \mathbf{S}$

$\\ \mathbf{A} \cdot \mathbf{V} \mathbf{V^{-1}} = \mathbf{S} \cdot \mathbf{V^{-1}}$

$\\ \mathbf{A} = \mathbf{S} \cdot \mathbf{V^{T}}$

Starting to look familiar isn't it? 

Remember, **any set of vectors (A) can be expressed in terms of their lengths of projections (S) on some set of orthogonal axes (V).** Let's dig into S a little more:

$\mathbf{S} = {\begin{bmatrix}s_{a1}&s_{a2}&\cdots \\ s_{b1}&s_{b2}&\cdots \\ \vdots&\vdots&\ddots \end{bmatrix}}$

You can see that each column vector of $\mathbf{S}$ represents the **lengths of projections of each point in A onto the corresponding orthogonal axes in V.** If we normalize the column vectors of $\mathbf{S}$, we can further decompose the matrix as follows:

$\mathbf{S} = \mathbf{U} \mathbf{\Sigma} ={\begin{bmatrix}\frac{s_{a1}}{\sigma_1}&\frac{s_{a2}}{\sigma_2}&\cdots \\ \frac{s_{b1}}{\sigma_1}&\frac{s_{b2}}{\sigma_2}&\cdots \\ \vdots&\vdots&\ddots \end{bmatrix}}
{\begin{bmatrix}\sigma_{1}&0&\cdots \\ 0&\sigma_{2}&\cdots \\ \vdots&\vdots&\ddots \end{bmatrix}} $

Where:
- $\mathbf{S}$ is an $m \times n$ matrix composed of the lengths of the projections 
- $\mathbf{U}$ is an $m \times n$ matrix composed of the normalized lengths of the projections (to unit vectors)
- $\mathbf{\Sigma}$ is a $n \times n$ matrix composed of the magnitudes of each column vector of $\mathbf{S}$, where $\sigma_i = \sqrt{(s_{ai})^2+(s_{bi})^2 +\cdots +(s_{mi})^2}$

Notice that the values of $\Sigma$ contain, by their definition, the sum of projection lengths onto a specific axis. Which means they literally **represent how close all the points are to a specific axis.** This is precisely why the SVD algorithm **orders eigenvalues from largest to smallest**. Essentially the larger the $\sigma$ value, the larger the variance that is captured by the corresponding axis **(more of the vector's direction is preserved after projection)**. Therefore we want to project out data onto said axis to capture that variance. 

So to recap, we can think of the SVD decomposition as follows:


$\mathbf{A} = \mathbf{U}\mathbf{\Sigma} \mathbf{V^{T}}$

where:
- $\mathbf{U}$ is an $m \times m$ orthogonal matrix, composed of normalized lengths of the projections onto the orthogonal axes of $\mathbf{V}$
- $\mathbf{\Sigma}$ an $m \times n$ diagonal matrix, composed of the projection lengths of each vector onto the orthogonal axes of $\mathbf{V}$. Note that we must pad the matrix with zeros at the bottom to make the matrix multiplication work. 
- $\mathbf{V}$ is a $n \times n$ matrix composed of the set of orthogonal unit vectors which our data is projected onto

Alternatively, recall that matrix products with a constant can also be broken down as follows:

$\mathbf{A} = \mathbf{U}\mathbf{\Sigma} \mathbf{V^{T}} = \sum_{i}{\sigma_i \cdot u_i  \cdot v_i^T } = \sigma_1 u_1 v_1^T + \sigma_2 u_2  v_2^T + \cdots + \sigma_p u_p  v_p^T  $

With this formulation, we can see that matrix $\mathbf{A}$ can be broken down into the summation of $p$ **rank 1** matrices, each of which is multiplied by its respective constant $\sigma_i$. So if we want to approximate $\mathbf{A}$ but we only get only get 1 $u$ vector and 1 $v$ vector, the best rank 1 approximation we can construct would be $\sigma_1 u_1 v_1^T$; if we get 2 vectors this would be $\sigma_1 u_1 v_1^T + \sigma_2 u_2  v_2^T $, and so on and so on. **This is simply because SVD arranged the eigenvalues (and corresponding eigenvectors) from highest to lowest.**

Now, to project our data onto our new axis, all we have to do is select the first two vectors in the matrix $\mathbf{V^T}$, and perform our usual projection operations. To illustrate, let's use the same dataset from earlier:

In [ ]:
# generating a dataset similar to Figure 8-2 in the textbook
np.random.seed(4)
m = 300
w1, w2, noise = 0.1, 0.3, 0.1
angles = np.random.rand(m) * 3 * np.pi / 2 - 0.
X = np.empty((m, 3))
X[:, 0] = np.cos(angles) + np.sin(angles)/2 + noise * np.random.randn(m) / 2
X[:, 1] = np.sin(angles) * 0.9 + noise * np.random.randn(m) / 2
X[:, 2] = X[:, 0] * w1 + X[:, 1] * w2 + noise*2 * np.random.randn(m)
X=X.T
print(X.shape)

# Plot data, subspace, and projection
fig = go.Figure(data=[go.Scatter3d(mode='markers', x=X[0,:],    y=X[1,:],    z=X[2,:], opacity=1.0, name='data')
                     ])
fig.update_traces(marker=dict(size=2))

In [ ]:
# let's see how this does on our earlier, arbitrarily-defined 2D subspace
v1 = np.array([1/3,2/3,2/3])
v2 = np.array([2/3,1/3,-2/3])
subspace_span = np.stack([v1,v2]).T

# Generate subspace sample
vectors = []
k = 1
for i in np.linspace(-k,k,50):
    for j in np.linspace(-k,k,50):
        vectors.append((subspace_span@np.array([i,j])))
subspace = np.stack(vectors)
proj = (subspace_span@subspace_span.T@X).T

# Plot data, subspace, and projection
fig = go.Figure(data=[go.Scatter3d(mode='markers', x=X[0,:],    y=X[1,:],    z=X[2,:], opacity=1.0, name='data'),
                      go.Scatter3d(mode='markers', x=subspace[:,0], y=subspace[:,1], z=subspace[:,2], opacity=0.4, name='subspace_V'),
                      go.Scatter3d(mode='markers', x=proj[:,0], y=proj[:,1], z=proj[:,2], opacity=0.6, name='projV_data')
                     ])
fig.update_traces(marker=dict(size=2))

# Normal vector
normal_vector = np.cross(subspace_span[:,0], subspace_span[:,1]) *0.5
normal_line = np.stack([normal_vector,normal_vector*-1])
colors = [[0, 'red']]
line_marker = dict(color='red', width=4)
fig.add_trace(go.Cone(x=[*normal_line[:,0]], y=[*normal_line[:,1]], z=[*normal_line[:,2]], u=[*normal_line[:,0]], v=[*normal_line[:,1]], w=[*normal_line[:,2]], sizeref=0.20, showscale=False, opacity=1, colorscale=colors))
fig.add_trace(go.Scatter3d(x=[*normal_line[:,0]], y=[*normal_line[:,1]], z=[*normal_line[:,2]], mode='lines', opacity=1, line=line_marker, name='normal_v'))
avg_dist = np.sqrt(np.mean((X - proj.T)**2))
print("Average distance to plane:",avg_dist)

fig.show()

In [ ]:
# now let's see what subspace we get by using SVD
U_, S_, V_ = np.linalg.svd(X.T, full_matrices=False)
subspace_span = V_.T[:,0:2]

# Generate subspace sample
vectors = []
k = 1
for i in np.linspace(-k,k,50):
    for j in np.linspace(-k,k,50):
        vectors.append((subspace_span@np.array([i,j])))
subspace = np.stack(vectors)
proj = (subspace_span@subspace_span.T@X).T


# Plot data, subspace, and projection
fig = go.Figure(data=[go.Scatter3d(mode='markers', x=X[0,:],    y=X[1,:],    z=X[2,:], opacity=1.0, name='data'),
                      go.Scatter3d(mode='markers', x=subspace[:,0], y=subspace[:,1], z=subspace[:,2], opacity=0.4, name='subspace_V'),
                      go.Scatter3d(mode='markers', x=proj[:,0], y=proj[:,1], z=proj[:,2], opacity=0.6, name='projV_data')
                     ])
fig.update_traces(marker=dict(size=2))

# Normal vector
normal_vector = np.cross(subspace_span[:,0], subspace_span[:,1]) *0.5
normal_line = np.stack([normal_vector,normal_vector*-1])
colors = [[0, 'red']]
line_marker = dict(color='red', width=4)
fig.add_trace(go.Cone(x=[*normal_line[:,0]], y=[*normal_line[:,1]], z=[*normal_line[:,2]], u=[*normal_line[:,0]], v=[*normal_line[:,1]], w=[*normal_line[:,2]], sizeref=0.20, showscale=False, opacity=1, colorscale=colors))
fig.add_trace(go.Scatter3d(x=[*normal_line[:,0]], y=[*normal_line[:,1]], z=[*normal_line[:,2]], mode='lines', opacity=1, line=line_marker, name='normal_v'))
avg_dist = np.sqrt(np.mean((X - proj.T)**2))
print("Average distance to plane:",avg_dist)
fig.show()

Remember we said above that $\mathbf{U} \cdot \mathbf{\Sigma}$ represents the magnitudes (lengths) of the original vectors projected onto the orthogonal basis vectors in $\mathbf{V}$:

$\mathbf{A} \cdot \mathbf{V} =  {\begin{bmatrix}a_{x}&a_{y}&\cdots \\ b_{x}&b_{y}&\cdots \\ \vdots&\vdots&\ddots \end{bmatrix}}
{\begin{bmatrix}v_{1x}&v_{2y}&\cdots \\ v_{1y}&v_{2y}&\cdots \\ \vdots&\vdots&\ddots \end{bmatrix}} = {\begin{bmatrix}s_{a1}&s_{a2}&\cdots \\ s_{b1}&s_{b2}&\cdots \\ \vdots&\vdots&\ddots \end{bmatrix}} = \mathbf{S}$
 

$\mathbf{S} = \mathbf{U} \mathbf{\Sigma} ={\begin{bmatrix}\frac{s_{a1}}{\sigma_1}&\frac{s_{a2}}{\sigma_2}&\cdots \\ \frac{s_{b1}}{\sigma_1}&\frac{s_{b2}}{\sigma_2}&\cdots \\ \vdots&\vdots&\ddots \end{bmatrix}}
{\begin{bmatrix}\sigma_{1}&0&\cdots \\ 0&\sigma_{2}&\cdots \\ \vdots&\vdots&\ddots \end{bmatrix}} $


Let's test this out with a few examples:

In [ ]:
# Project onto V to get the lengths
X.T[0:5].reshape((5,3))@V_.T

In [ ]:
# Get vector lengths from U@Sigma
((U_@np.diag(S_)))[0:5]

Nice! We can also see that the norms of the columns of $\mathbf{U} \cdot \mathbf{\Sigma}$ should equal the values of $\mathbf{\Sigma}$:

In [ ]:
S_

In [ ]:
# eigenvalues
print(np.linalg.norm(U_@np.diag(S_)[:,0]))
print(np.linalg.norm(U_@np.diag(S_)[:,1]))
print(np.linalg.norm(U_@np.diag(S_)[:,2]))

$\textbf{proj}_{\textbf{V}}{\vec{x}} = \matrix{A}(\matrix{A}^\top \matrix{A})^{-1} \matrix{A}^\top \vec{x}$


### Economy vs Full SVD



There is one small caveat to address. It is typical for a high dimensional dataset to lie close to a hyperplane in a subspace with fewer dimensions (as we saw with our dataset above). In these cases, the *economy* SVD will return only the *thin* SVD matrices corresponding to the rank of the data matrix. 

So if you have a $m \times n$ matrix $\mathbf{A}$ of rank $p$, the economy SVD will produce:
$\underset{m \times n}{\mathbf{A}} = \underset{m \times p}{\mathbf{U}} \cdot \underset{p \times p}{\mathbf{\Sigma}} \cdot \underset{p \times n}{\mathbf{V^T}}$

Similarly, the Full SVD would be:
$\underset{m \times n}{\mathbf{A}} = \underset{m \times m}{\mathbf{U}} \cdot \underset{m \times n}{\mathbf{\Sigma}} \cdot \underset{n \times n}{\mathbf{V^T}}$

As you may have noticed, because we will always only have $p$ **singular values**, to run the Full SVD we have to pad our diagonal matrix $\mathbf{\Sigma}$ with $m-p$ rows of zeros and $n-p$ columns of zeros. We also have to generate additional orthogonal vectors for $\mathbf{V}$ and $\mathbf{U}$ using the respective Null Spaces and the the **Gram-Schmidt process:**

\begin{align}
  \mathbf{A} &=
  \mathbf{U} \, \Sigma \, \mathbf{V}^{T} \\
%
 &=
% U 
  \left[ \begin{array}{cc}
     \color{blue}{\mathbf{U}_{\mathcal{R}}} & \color{red}{\mathbf{U}_{\mathcal{N}}}
  \end{array} \right]  
% Sigma
  \left[ \begin{array}{cccc|cc}
     \sigma_{1} & 0 & \dots &  &   & \dots &  0 \\
     0 & \sigma_{2}  \\
     \vdots && \ddots \\
       & & & \sigma_{\rho} \\\hline
       & & & & 0 & \\
     \vdots &&&&&\ddots \\
     0 & & &   &   &  & 0 \\
  \end{array} \right]
% V 
  \left[ \begin{array}{c}
     \color{blue}{\mathbf{V}_{\mathcal{R}}}^{T} \\ 
     \color{red}{\mathbf{V}_{\mathcal{N}}}^{T}
  \end{array} \right]  \\
%
  & =
% U
   \left[ \begin{array}{cccccccc}
    \color{blue}{u_{1}} & \dots & \color{blue}{u_{\rho}} & \color{red}{u_{\rho+1}} & \dots & \color{red}{u_{m}}
  \end{array} \right]
% Sigma
  \left[ \begin{array}{cc}
     \mathbf{S}_{\rho\times \rho} & \mathbf{0} \\
     \mathbf{0} & \mathbf{0} 
  \end{array} \right]
% V
   \left[ \begin{array}{c}
    \color{blue}{v_{1}^{T}} \\ 
    \vdots \\
    \color{blue}{v_{\rho}^{T}} \\
    \color{red}{v_{\rho+1}^{T}} \\
    \vdots \\ 
    \color{red}{v_{n}^{T}}
  \end{array} \right]
%
\end{align}

You also may have noticed that when we calculate the Full SVD, the additional vectors generated from the null spaces get zeroed out by our $\mathbf{\Sigma}$ matrix. So generally speaking you will probably never want to compute the Full SVD.


- For a more complete walk through of the Full SVD, see this excellent *stack exhange* post: https://math.stackexchange.com/questions/1771013/how-is-the-null-space-related-to-singular-value-decomposition
- Also check out this article for further intuition behind SVD: https://jonathan-hui.medium.com/machine-learning-singular-value-decomposition-svd-principal-component-analysis-pca-1d45e885e491#:~:text=What%20is%20the%20difference%20between,PCA%20skips%20less%20significant%20components.



In [ ]:
# economy SVD
np.random.seed(4)
m = 300
w1, w2, noise = 0.1, 0.3, 0.1
angles = np.random.rand(m) * 3 * np.pi / 2 - 0.
X = np.empty((m, 3))
X[:, 0] = np.cos(angles) + np.sin(angles)/2 + noise * np.random.randn(m) / 2
X[:, 1] = np.sin(angles) * 0.9 + noise * np.random.randn(m) / 2
X[:, 2] = X[:, 0] * w1 + X[:, 1] * w2 + noise*2 * np.random.randn(m)
X=X.T

U_, S_, V_ = np.linalg.svd(X.T, full_matrices=False)
for i in U_, S_, V_:
    print(i.shape)


In [ ]:
# Full SVD
U_, S_, V_ = np.linalg.svd(X.T, full_matrices=True)
for i in U_, S_, V_:
    print(i.shape)


## PCA using SVD

Now that we understand how SVD works, you can clearly see how it relates to Principal Component Analysis, i.e. **identifying the axis (hyperplane) that accounts for the largest amount of variance in the training set, and then projecting the data onto that hyperplane**.

In fact, if you look at the `sklearn` documentation for `PCA`, you will find the following:

```Principal component analysis (PCA). Linear dimensionality reduction using Singular Value Decomposition of the data to project it to a lower dimensional space. The input data is centered but not scaled for each feature before applying the SVD.```

So to replicate this approach, we first need to centre the data, and then apply singular value decomposition just as we did above. Once you have identified all the principal components, you can reduce the dimensionality of the dataset down to $d$ dimensions by projecting it onto the hyperplane defined by the first $d$ principal components. 

Selecting this hyperplane ensures that the projection will preserve as much variance as possible. 

To project the training set onto the hyperplane and obtain a reduced dataset $X_{d-proj}$ of dimensionality d, compute the matrix multiplication of the training set matrix X by the matrix $W_d$, defined as the matrix containing the first $d$ columns of $V$:



In [ ]:
# Generate dataset
np.random.seed(4)
m = 300
w1, w2, noise = 0.1, 0.3, 0.1
angles = np.random.rand(m) * 3 * np.pi / 2 - 0.5
X = np.empty((m, 3))
X[:, 0] = np.cos(angles) + np.sin(angles)/2 + noise * np.random.randn(m) / 2
X[:, 1] = np.sin(angles) * 0.9 + noise * np.random.randn(m) / 2
X[:, 2] = X[:, 0] * w1 + X[:, 1] * w2 + noise*2 * np.random.randn(m)
print(X.shape)

# Centre data
X_centered = X - X.mean(axis=0) #sklearns PCA should centre data for you
U, S, V = np.linalg.svd(X_centered, full_matrices=False)
print(X_centered.shape)
print(U.shape)
print(S.shape)
print(V.shape)
c1 = V.T[:, 0]
c2 = V.T[:, 1]

print(c1, c2)

In [ ]:
# Get components
W2 = V.T[:, :2]
W2

In [ ]:
# Dot product to project onto 2 dimensions
X2D = X_centered.dot(W2)
X2D_using_svd = X2D
X2D_using_svd.shape

In [ ]:
plt.scatter(X2D_using_svd[:,0], X2D_using_svd[:,1]);

SVD recovers the principal axes of maximum variance.

## PCA using Scikit-Learn

With Scikit-Learn, PCA is really trivial. It even takes care of mean centering for you:

In [ ]:
import sklearn.decomposition as decomp
dir(decomp)

In [ ]:
X.shape

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components = 2)
X2D = pca.fit_transform(X)
print(X2D.shape)

In [ ]:
X2D[:5]

In [ ]:
X2D_using_svd[:5]

After fitting the PCA transformer to the dataset, its `components_` attribute holds the transpose of $W_d$ (e.g., the unit vector that defines the first principal component is equal to `pca.components_.T[:, 0]`). 


In [ ]:
pca.components_

In [ ]:
W2.T

In [ ]:
np.allclose(X2D, X2D_using_svd)

## Explained Variance Ratio

Another useful piece of information is the **explained variance ratio** of each principal component, available via the `explained_variance_ratio_` variable. 

The ratio indicates the **proportion of the dataset’s variance that lies along each principal component**. 

In [ ]:
pca.explained_variance_ratio_

In [ ]:
1 - pca.explained_variance_ratio_.sum()

The first dimension explains 79.0% of the variance, while the second explains 17.9%. By projecting down to 2D, we lost about 3% of the variance.

Here is how to compute the explained variance ratio using the SVD approach (recall that `s` is the diagonal of the matrix `S`):


In [ ]:
X_centered = X - X.mean(axis=0) #sklearns PCA should centre data for you
U, s, Vt = np.linalg.svd(X_centered)
m, n = X.shape


np.square(s) / np.square(s).sum()

Instead of arbitrarily choosing the number of dimensions to reduce down to, it is simpler to choose the number of dimensions that add up to a sufficiently large portion of the variance (e.g., 95%). 

The following code performs PCA without reducing dimensionality, then computes the minimum number of dimensions required to preserve 95% of the training set’s variance:
    

In [ ]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1)
mnist.target = mnist.target.astype(np.uint8)

In [ ]:
from sklearn.model_selection import train_test_split

X = mnist["data"]
y = mnist["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y)

In [ ]:
X_train = X_train.values

In [ ]:
X_train.shape

In [ ]:
pca = PCA()
pca.fit(X_train)
var = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(var >= 0.95) + 1
d

In [ ]:
var[0:5]

You can do the same with sklearn, using the `n_components` argument:

In [ ]:
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X_train)

In [ ]:
pca.n_components_

In [ ]:
np.sum(pca.explained_variance_ratio_)

In [ ]:
var

Yet another option is to plot the explained variance as a function of the number of dimensions. 

There will usually be an **elbow in the curve, where the explained variance stops growing fast.**

In this case, you can see that reducing the dimensionality down to about 100 dimensions wouldn’t lose too much explained variance. 

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(var, linewidth=3)
plt.axis([0, 400, 0, 1])
plt.xlabel("Dimensions")
plt.ylabel("Explained Variance")
plt.plot([d, d], [0, 0.95], "k:")
plt.plot([0, d], [0.95, 0.95], "k:")
plt.plot(d, 0.95, "ko")
plt.annotate("Elbow", xy=(65, 0.85), xytext=(70, 0.7),
             arrowprops=dict(arrowstyle="->"), fontsize=16)
plt.grid(True)
plt.show()

## PCA for compression

After dimensionality reduction, the training set takes up much less space. As an example, try applying PCA to the MNIST dataset while preserving 95% of its variance. You should find that each instance will have just over 150 features, instead of the original 784 features. 
- So, while most of the variance is preserved, the dataset is now less than 20% of its original size! 
- This is a reasonable compression ratio, and you can see how this size reduction can speed up a classification algorithm (such as an SVM classifier) tremendously. 

It is also possible to **decompress the reduced dataset back to 784 dimensions by applying the inverse transformation of the PCA projection.**
- This won’t give you back the original data, since the projection lost a bit of information (within the 5% variance that was dropped), but it will likely be close to the original data.
- The mean squared distance between the original data and the reconstructed data (compressed and then decompressed) is called the **reconstruction error.** 

In [ ]:
pca = PCA(n_components = 154)
X_reduced_pca = pca.fit_transform(X_train)
X_recovered = pca.inverse_transform(X_reduced_pca)

In [ ]:
def plot_digits(instances, images_per_row=5, **options):
    size = 28
    images_per_row = min(len(instances), images_per_row)
    images = [instance.reshape(size,size) for instance in instances]
    print(images[0].shape)
    n_rows = (len(instances) - 1) // images_per_row + 1
    row_images = []
    n_empty = n_rows * images_per_row - len(instances)
    images.append(np.zeros((size, size * n_empty)))
    for row in range(n_rows):
        rimages = images[row * images_per_row : (row + 1) * images_per_row]
        row_images.append(np.concatenate(rimages, axis=1))
    image = np.concatenate(row_images, axis=0)
    plt.imshow(image, cmap = mpl.cm.binary, **options)
    plt.axis("off")

In [ ]:
plt.figure(figsize=(7, 4))
plt.subplot(121)
plot_digits(X_train[::2100])
plt.title("Original", fontsize=16)
plt.subplot(122)
plot_digits(X_recovered[::2100])
plt.title("Compressed", fontsize=16);

The equation of the inverse transformation is defined as follows:

$$\mathbf{X}_{recovered} = \mathbf{X}_{d-proj}\mathbf{W}_d^T$$

## Incremental PCA

One problem with the preceding implementations of PCA is that they require **the whole training set to fit in memory in order for the algorithm to run.** 

Fortunately, Incremental PCA (IPCA) algorithms have been developed. 
- IPCA allows you to split the training set into **mini-batches** and feed the algorithm one mini-batch at a time. 
- This is useful for large training sets and for applying PCA online. 

The following code splits the MNIST dataset into 100 mini-batches (using NumPy’s array_split() function) and feeds them to Scikit-Learn’s IncrementalPCA class to reduce the dimensionality of the MNIST dataset down to 154 dimensions (just like before). 

**NOTE:** Note that you must call the `partial_fit()` method with each mini-batch, rather than the `fit()` method with the whole training set: 


In [ ]:
from sklearn.decomposition import IncrementalPCA

n_batches = 100
inc_pca = IncrementalPCA(n_components=154)
for X_batch in np.array_split(X_train, n_batches):
    print(".", end="") # not shown in the book
    inc_pca.partial_fit(X_batch)

X_reduced_inc_pca = inc_pca.transform(X_train)

In [ ]:
X_recovered_inc_pca = inc_pca.inverse_transform(X_reduced_inc_pca)

In [ ]:
plt.figure(figsize=(7, 4))
plt.subplot(121)
plot_digits(X_train[::2100])
plt.subplot(122)
plot_digits(X_recovered_inc_pca[::2100])
plt.tight_layout()

Let's compare the results of transforming MNIST using regular PCA and incremental PCA. First, the means are equal: 

In [ ]:
np.allclose(pca.mean_, inc_pca.mean_)

But the results are not exactly identical. Incremental PCA gives a very good approximate solution, but it's not perfect:

In [ ]:
np.allclose(X_reduced_pca, X_reduced_inc_pca)

### Using `memmap()`

Alternatively, you can use NumPy’s memmap class, which allows you to **manipulate a large array stored in a binary file on disk as if it were entirely in memory**. 

The class loads only the data it needs in memory, **when it needs it.** 

Since the IncrementalPCA class uses only a small part of the array at any given time, the memory usage remains under control. This makes it possible to call the usual `fit()` method, as you can see in the following code: 

In [ ]:
filename = "my_mnist.data"
m, n = X_train.shape

X_mm = np.memmap(filename, dtype='float32', mode='write', shape=(m, n))
X_mm[:] = X_train

del X_mm

In [ ]:
X_mm = np.memmap(filename, dtype="float32", mode="readonly", shape=(m, n))

batch_size = m // n_batches
inc_pca = IncrementalPCA(n_components=154, batch_size=batch_size)
inc_pca.fit(X_mm)

In [ ]:
rnd_pca = PCA(n_components=154, svd_solver="randomized", random_state=42)
X_reduced = rnd_pca.fit_transform(X_train)

# Kernel PCA

In Module 5 we discussed the **kernel trick**, a mathematical technique that implicitly maps instances into a very high-dimensional space (called the feature space ), enabling nonlinear classification and regression with Support Vector Machines. 

Recall that a **linear decision boundary in the high-dimensional feature space corresponds to a complex nonlinear decision boundary in the original space**:
- It turns out that the same trick can be applied to PCA, making it possible to perform complex nonlinear projections for dimensionality reduction. 
- This is called Kernel PCA (kPCA) .
- It is often good at preserving clusters of instances after projection, or sometimes even unrolling datasets that lie close to a twisted manifold. 

The following code uses Scikit-Learn’s KernelPCA class to perform kPCA with an RBF kernel:

In [ ]:
from sklearn.datasets import make_swiss_roll
import plotly.graph_objects as go

X, t = make_swiss_roll(n_samples=1000, noise=0.2, random_state=42)

params = dict(size=4,color=t,colorscale='Viridis')
fig = go.Figure(data=[go.Scatter3d(x=X[:,0],
                                   y=X[:,1],
                                   z=X[:,2],
                                   mode='markers',
                                   marker=params
                                  )])

fig.show()

In [ ]:
from sklearn.decomposition import KernelPCA

rbf_pca = KernelPCA(n_components = 2, kernel="rbf", gamma=0.04)
X_reduced = rbf_pca.fit_transform(X)
print(X_reduced.shape)
plt.scatter(X_reduced[:,0], X_reduced[:,1], c=t);

In [ ]:
from sklearn.decomposition import KernelPCA

lin_pca = KernelPCA(n_components = 2, kernel="linear", fit_inverse_transform=True)
rbf_pca = KernelPCA(n_components = 2, kernel="rbf", gamma=0.0433, fit_inverse_transform=True)
sig_pca = KernelPCA(n_components = 2, kernel="cosine", gamma=0.001, coef0=1, fit_inverse_transform=True)

y = t > 6.9

plt.figure(figsize=(11, 4))
for subplot, pca, title in ((131, lin_pca, "Linear kernel"), (132, rbf_pca, "RBF kernel, $\gamma=0.04$"), (133, sig_pca, "Sigmoid kernel, $\gamma=10^{-3}, r=1$")):
    X_reduced = pca.fit_transform(X)
    if subplot == 132:
        X_reduced_rbf = X_reduced
    
    plt.subplot(subplot)
    #plt.plot(X_reduced[y, 0], X_reduced[y, 1], "gs")
    #plt.plot(X_reduced[~y, 0], X_reduced[~y, 1], "y^")
    plt.title(title, fontsize=14)
    plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=t, cmap=plt.cm.hot)
    plt.xlabel("$z_1$", fontsize=18)
    if subplot == 131:
        plt.ylabel("$z_2$", fontsize=18, rotation=0)
    plt.grid(True)

plt.show()

## Kernel Selection
Selecting a Kernel and Tuning Hyperparameters As kPCA is an unsupervised learning algorithm, there is **no obvious performance measure to help you select the best kernel and hyperparameter values**. 

That said, dimensionality reduction is often a **preparation step for a supervised learning task (e.g., classification).**
- You can use `gridsearch` to select the kernel and hyperparameters that lead to the best performance on that task. 

The following code creates a two-step pipeline:
- First reducing dimensionality to two dimensions using kPCA 
- Then applying Logistic Regression for classification 
- Then it uses GridSearchCV to find the best kernel and gamma value for kPCA in order to get the best classification accuracy at the end of the pipeline 


In [ ]:
X

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

clf = Pipeline([
        ("kpca", KernelPCA(n_components=2)),
        ("log_reg", LogisticRegression(solver="lbfgs"))
    ])

param_grid = [{
        "kpca__gamma": np.linspace(0.03, 0.05, 10),
        "kpca__kernel": ["rbf", "sigmoid"]
    }]

grid_search = GridSearchCV(clf, param_grid, cv=3)
grid_search.fit(X, y)

In [ ]:
print(grid_search.best_params_)

## Reconstruction error

Another approach, this time entirely unsupervised, is to select the kernel and hyperparameters that yield **the lowest reconstruction error.**

Note that reconstruction is not as easy as with linear PCA. The figure below shows the original Swiss roll 3D dataset (top left) and the resulting 2D dataset after kPCA is applied using an RBF kernel (top right). 

<img src="images_module07/08_kpca.png" width="500"/>

Thanks to the kernel trick, this transformation is **mathematically equivalent** to using the feature map φ to map the training set to an infinite-dimensional feature space (bottom right), then projecting the transformed training set down to 2D using linear PCA. 

Notice that if we could invert the linear PCA step for a given instance in the reduced space, the reconstructed point would lie in feature space, not in the original space (e.g., like the one represented by an X in the diagram). Since the feature space is infinite-dimensional, we cannot compute the reconstructed point, and therefore we cannot compute the true reconstruction error. 

Fortunately, it is possible to find a point in the original space that would map close to the reconstructed point. This point is called the reconstruction pre-image . Once you have this pre-image, you can measure its squared distance to the original instance. You can then select the kernel and hyperparameters that minimize this reconstruction pre-image error. 

You may be wondering how to perform this reconstruction. One solution is to train a supervised regression model, with the projected instances as the training set and the original instances as the targets. Scikit-Learn will do this automatically if you set `fit_inverse_transform=True` , as shown in the following code: 

In [ ]:
rbf_pca = KernelPCA(n_components = 2, kernel="rbf", gamma=0.0433,
                    fit_inverse_transform=True)
X_reduced = rbf_pca.fit_transform(X)
X_preimage = rbf_pca.inverse_transform(X_reduced)

In [ ]:
from sklearn.metrics import mean_squared_error

mean_squared_error(X, X_preimage)

# LLE

Locally Linear Embedding (LLE) is another powerful nonlinear dimensionality reduction (NLDR) technique. It is a **Manifold Learning technique** that does not rely on projections, like the previous algorithms do. 

In a nutshell, LLE works by first **measuring how each training instance linearly relates to its closest neighbors (c.n.)**, and then looking for a low-dimensional representation of the training set where these local relationships are best preserved (more details shortly). 

This approach makes it particularly good at **unrolling twisted manifolds**, especially when there is not too much noise. The following code uses Scikit-Learn’s `LocallyLinearEmbedding` class to unroll the swiss roll:

In [ ]:
X, t = make_swiss_roll(n_samples=1000, noise=0.2, random_state=41)
X.shape

In [ ]:
params = dict(size=4,color=t,colorscale='Viridis')
fig = go.Figure(data=[go.Scatter3d(x=X[:,0],
                                   y=X[:,1],
                                   z=X[:,2],
                                   mode='markers',
                                   marker=params
                                  )])

fig.show()

In [ ]:
from sklearn.manifold import LocallyLinearEmbedding

lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10, random_state=42)
X_reduced = lle.fit_transform(X)

In [ ]:
plt.title("Unrolled swiss roll using LLE", fontsize=14)
plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=t, cmap=plt.cm.hot)
plt.xlabel("$z_1$", fontsize=18)
plt.ylabel("$z_2$", fontsize=18)
plt.axis([-0.065, 0.055, -0.1, 0.12])
plt.grid(True)

plt.show()

As you can see, the Swiss roll is completely unrolled, and the distances between instances are locally well preserved. 

However, distances are not preserved on a larger scale: 
- the left part of the unrolled Swiss roll is stretched
- and the right part is squeezed. 

Nevertheless, LLE did a pretty good job at modeling the manifold.



Here’s how LLE works: 
- For each training instance $x ( i )$ , the algorithm identifies its $k$ closest neighbors (in the preceding code k = 10)
- Then it tries to reconstruct $x ( i )$ as a linear function of these neighbors. 
- More specifically, it finds the weights $w_{i,j}$ such that the squared distance between $x^{(i)}$ and $\sum_{j=1}^m w_{i,j}x^{(j)} $ is as small as possible (assuming $w_{ij} = 0$ if $x^{(j)}$ is not one of the k closest neighbors of $x^{(i)}$)

Thus the first step of LLE is the constrained optimization problem:

<img src="images_module07/09_lle.png" width="500"/>



After this step, the weight matrix (containing the weights ) **encodes the local linear relationships between the training instances.** 

The second step is to map the training instances into a d dimensional space (where d < n ) while preserving these local relationships as much as possible. 

- If $z ( i )$ is the image of $x ( i )$ in this d dimensional space, then we want the squared distance between $z ( i )$  and $\sum_{j=1}^m w_{i,j}x^{(j)}$ to be as small as possible. 

This idea leads to the unconstrained optimization problem described below:
- It looks very similar to the first step, but instead of keeping the instances fixed and finding the optimal weights, we are doing the reverse
- keeping the weights fixed and finding the optimal position of the instances’ images in the low-dimensional space. Note that Z is the matrix containing all $z ( i )$. 

<img src="images_module07/10_lle.png" width="500"/>

Here is a great breakdown of the steps in pseudocode: https://cs.nyu.edu/~roweis/lle/algorithm.html


In [ ]:
# ?LocallyLinearEmbedding

# t-SNE

t-SNE (t-distributed Stochastic Neighbor Embedding) is a **nonlinear dimensionality reduction technique** well-suited for embedding high-dimensional data for visualization in a low-dimensional space of two or three dimensions. That said, it is mostly used for visualization and not data pre-processing (like you might with PCA).
- It’s extremely common to take the features from an inner layer of a deep learning model and plot them in 2-dimensions using t-SNE to reduce the dimensionality. 

The t-SNE algorithm comprises two main stages:
- First, t-SNE constructs a probability distribution over pairs of high-dimensional objects in such a way that similar objects are assigned a higher probability while dissimilar points are assigned a lower probability.
- Second, t-SNE defines a similar probability distribution over the points in the low-dimensional map, and it minimizes the Kullback–Leibler divergence (KL divergence) between the two distributions with respect to the locations of the points in the map (https://en.wikipedia.org/wiki/Kullback%E2%80%93Leibler_divergence)
- Note that while the original algorithm uses the Euclidean distance between objects as the base of its similarity metric, this can be changed as appropriate.

t-SNE has a hyper-parameter called **perplexity:** Perplexity balances the attention t-SNE gives to local and global aspects of the data and can have large effects on the resulting plot. A few notes on this parameter:
- It is roughly a **guess of the number of close neighbors each point has.** Thus, a denser dataset usually requires a higher perplexity value.
- It is recommended to be between 5 and 50.
- It should be smaller than the number of data points.
- The biggest mistake people make with t-SNE is only using one value for perplexity and not testing how the results change with other values. If choosing different values between 5 and 50 significantly change your interpretation of the data, then you should consider other ways to visualize or validate your hypothesis.

For more details, see this post: https://www.tivadardanka.com/blog/how-tsne-works

In [ ]:
X, t = make_swiss_roll(n_samples=1000, noise=0.2, random_state=41)
X.shape

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(perplexity=50, n_components=2, random_state=42)
X_reduced_tsne = tsne.fit_transform(X)

In [ ]:
X_reduced_tsne.shape

In [ ]:
params = dict(size=4,color=t,colorscale='Viridis')
fig = go.Figure(data=[go.Scatter3d(x=X[:,0],
                                   y=X[:,1],
                                   z=X[:,2],
                                   mode='markers',
                                   marker=params
                                  )])

fig.show()

In [ ]:
params = dict(size=4,color=t,colorscale='Viridis')
fig = go.Figure(data=[go.Scatter(x=X_reduced_tsne[:,0],
                                   y=X_reduced_tsne[:,1],
                                   mode='markers',
                                   marker=params
                                  )])

fig.show()

In [ ]:
# ?tsne

# MDS and Isomap

In [ ]:
from sklearn.manifold import MDS

mds = MDS(n_components=2, random_state=42)
X_reduced_mds = mds.fit_transform(X)

In [ ]:
from sklearn.manifold import Isomap

isomap = Isomap(n_components=2)
X_reduced_isomap = isomap.fit_transform(X)

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42)
X_reduced_tsne = tsne.fit_transform(X)

In [ ]:
titles = ["MDS", "Isomap", "t-SNE"]

plt.figure(figsize=(11,4))

for subplot, title, X_reduced in zip((131, 132, 133), titles,
                                     (X_reduced_mds, X_reduced_isomap, X_reduced_tsne)):
    plt.subplot(subplot)
    plt.title(title, fontsize=14)
    plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=t, cmap=plt.cm.hot)
    plt.xlabel("$z_1$", fontsize=18)
    if subplot == 131:
        plt.ylabel("$z_2$", fontsize=18, rotation=0)
    plt.grid(True)

plt.show()

## Visualizing MNIST in 3D
For fun, lets reduce MNIST down to 3 dimensions and visualize the results with plotly:

In [ ]:
# Get data
#mnist = fetch_openml('mnist_784', version=1)
#mnist.target = mnist.target.astype(np.uint8)

from sklearn.model_selection import train_test_split

X = mnist["data"]
y = mnist["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.02)

print(X_test.shape)
print(y_test.shape)

In [ ]:
%%time

from sklearn.manifold import Isomap
from sklearn.manifold import MDS
from sklearn.manifold import TSNE
from sklearn.manifold import LocallyLinearEmbedding

reduce = Isomap(n_components=3)
#reduce = MDS(n_components=3, random_state=42)
#reduce = TSNE(n_components=3, random_state=42)
#reduce = LocallyLinearEmbedding(n_components=3, n_neighbors=20, random_state=42)


mnist_reduced = reduce.fit_transform(X_test)
mnist_reduced.shape

In [ ]:
import plotly.graph_objects as go

params = dict(size=6,color=y_test,colorscale='Viridis')
fig = go.Figure(data=[go.Scatter3d(x=mnist_reduced[:,0], y=mnist_reduced[:,1], z=mnist_reduced[:,2], mode='markers', marker=params, hovertext=y_test,)])

fig.update_layout(
    autosize=False,
    width=900,
    height=700,
    margin=dict(l=50, r=50, b=100, t=100,pad=2), 
    paper_bgcolor="LightSteelBlue")
fig.show()

# Word Vectors

https://towardsdatascience.com/understanding-word2vec-embedding-in-practice-3e9b8985953

Link to download glove embeddings: https://nlp.stanford.edu/projects/glove/

In [ ]:
# Import pretrained word vectors
import os
import numpy as np

path_to_wemb = "glove.6B.200d.txt"
embedding_index = {}
embeddings = []

f = open(path_to_wemb)
count = 0
for line in f:
    values = line.split()
    # Update index
    word = values[0]
    embedding_index[word] = count
    # Get coefs
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings.append(coefs)
    count += 1
    if count % 100000 == 0:
        print(f"Completed {count} words...")

len(embedding_index)

In [ ]:
list(embedding_index.items())[100:110]

In [ ]:
len(embeddings)

In [ ]:
embs_np = np.stack(embeddings)
embs_np.shape

In [ ]:
animals = ['dog', 'cat', 'hamster', 'bunny', 'frog', 'shark','puppy','lion', 'toad','monkey']
names = ['john','frank', 'mike','jason','cindy','hailey','christine','samantha', 'carl', 'mohamed']
cities = ['toronto', 'edmonton', 'vancouver', 'shanghai','paris', 'dallas', 'pittsburgh','chicago','calgary','london']
countries = ['america','canada', 'china', 'australia','japan','india','pakistan','france','mexico','greece']
food = ['pizza', 'marshmallow','spaghetti', 'dumplings', 'corn', 'chicken', 'beef', 'tacos','bread','sandwich']
furniture = ['chair', 'table', 'stool','desk','carpet','couch','nightstand','wardrobe','dresser', 'ottoman']
fruit = ['apple','banana','orange','grape','tomato','melon','pineapple','kiwi','strawberry','blackberry']
labels = [*animals, *names, *cities,*countries, *food , *furniture, *fruit]

colors = [1]*len(animals) + [2]*len(names) + [3]*len(cities) + [4]*len(countries) + [5]*len(food) + [6]*len(furniture) + [7]*len(fruit) 
len(labels)

In [ ]:
items = [animals, names, cities,countries, food, furniture, fruit]
embs = np.stack([embeddings[embedding_index[word]] for l in items for word in l])
embs.shape

In [ ]:
%%time

from sklearn.manifold import Isomap
from sklearn.manifold import MDS
from sklearn.manifold import TSNE
from sklearn.manifold import LocallyLinearEmbedding

reduce = Isomap(n_components=3)
#reduce = MDS(n_components=3, random_state=42)
#reduce = TSNE(n_components=3, random_state=42, perplexity=10)
#reduce = LocallyLinearEmbedding(n_components=3, n_neighbors=10, random_state=42)


wv_reduced = reduce.fit_transform(embs)
wv_reduced.shape

In [ ]:
import plotly.graph_objects as go

params = dict(size=6,color=colors,colorscale='Viridis')
fig = go.Figure(data=[go.Scatter3d(x=wv_reduced[:,0], y=wv_reduced[:,1], z=wv_reduced[:,2], mode='markers', marker=params, hovertext=labels)])
fig.update_layout(
    autosize=False,
    width=900,
    height=700,
    margin=dict(l=50, r=50, b=100, t=100, pad=2),
    paper_bgcolor="LightSteelBlue",
)

fig.show()

# Semantic Similarity

https://medium.com/@adriensieg/text-similarities-da019229c894

Advanced:

https://medium.com/analytics-vidhya/semantic-similarity-in-sentences-and-bert-e8d34f5a4677

https://towardsdatascience.com/use-cases-of-googles-universal-sentence-encoder-in-production-dd5aaab4fc15

## Setup

This section covers the basic steps to determining the **similarity between two sentences** using a natural language processing module called `spaCy`. 

We know that words can be represented as vectors that are generated using an algorithm called word2vec, which can be trained using any open-sources libraries such as Gensim or FastText.

Fortunately, spaCy has its own words vectors built-in that are ready to be used (only applicable for certain language and models).

To get started, download spacy:

    conda install -c conda-forge spacy -y
    
And then download the language models (the word vectors):

    python -m spacy download en_core_web_lg



## Importing and loading module

In [ ]:
# conda install spacy

In [ ]:
!python -m spacy download en_core_web_lg

In [ ]:
import spacy
nlp = spacy.load("en_core_web_lg")

stop_words = spacy.lang.en.stop_words.STOP_WORDS
len(stop_words)

In [ ]:
dir(nlp.Defaults)

In [ ]:
doc = nlp(u"My name is Mike and I love data science and apples and Tim Cook. He works for Apple Corporation. John Cena is my hero")

doc.ents

In [ ]:
doc.vector.shape

In [ ]:
list(doc.noun_chunks)

In [ ]:
[token.text for token in doc]

In [ ]:
tokens = doc.to_json()['tokens']
tokens[0:5]

## Cosine similarity

\begin{equation}
\cos (\theta)= {{\bf t} {\bf e} \over \|{\bf t}\| \|{\bf e}\|} = \frac{ \sum_{i=1}^{n}{{\bf t}_i{\bf e}_i} }{ \sqrt{\sum_{i=1}^{n}{({\bf t}_i)^2}} \sqrt{\sum_{i=1}^{n}{({\bf e}_i)^2}} }
\end{equation}

- $-1$ means vectors are pointing in opposite direction
- $+1$ means vectors are pointing in the same direction
- $0$ means vectors are orthogonal

In [ ]:
def cosine_sim(v1, v2):
    return v1.dot(v2) / (np.linalg.norm(v1)*np.linalg.norm(v2))

In [ ]:
# Cosine=-1
a = np.array([2,1])
b = np.array([-2,-1])
# Cosine=1
#a = np.array([-4,-2])
#b = np.array([-2,-1])
# 90 deg
#a = np.array([2,0])
#b = np.array([0,2])


plt.scatter([a[0], b[0]], [a[1], b[1]])
plt.grid()
plt.axvline(0,c='black')
plt.axhline(0,c='black')
plt.xlim(-5,5)
plt.ylim(-5,5)

print(cosine_sim(a,b))

In [ ]:
doc1 = nlp("How do I turn sound on/off?")
doc2 = nlp("How do I turn up the volume?")
doc3 = nlp("Mary hates avocados, mainly because they go bad so quickly?")
doc4 = nlp("I love data!")

print(doc.vector.shape)
#?doc1.similarity

In [ ]:
cosine_sim(doc1.vector, doc2.vector)

In [ ]:
doc1.similarity(doc2)

In [ ]:
doc1.similarity(doc3)

In [ ]:
doc1.similarity(doc4)

In [ ]:
doc_a = nlp("How do I turn up the volume")
doc_b = nlp("volume how up the turn I do")

doc_a.similarity(doc_b)

## Sentence similarity with W2V avergaes

In [ ]:
sents = [
    ### NEGATIVE
    "pets are the worst, I never want to get one",
    "I hate dogs, they always smell bad",
    "Dogs are gross!",
    "Dogs are fine, but cats are the worst",
    "Cats and dogs both suck equally",
    "I never ever ever want to get a dog",
    "I never ever ever want to get a cat",
    "Cats are too loud and I'm alergic!",
    "Dogs are too loud and they smeel bad",
    "I don't want to have to walk the dog everyday",
    "Pets are simply too much work",
    ### POSITIVE
    "pets are the best, I really want to get one",
    "I love dogs, I want to get a puppy so bad!",
    "I would happily get a cat or a dog",
    "I hope we get a puppy soon",
    "I love cats, but I love dogs even more",
    "I can't wait to get a puppy!",
    "I love taking my dog for walks",
    "I love to pet my cat and give it treats",
    "My dog is so freindly, hes my best friend",
    "My cat is so cute!",
    "My dog is so playful",
    # RANDOM
    "Can you pass me the salt?",
    "I can't wait for the weekend",
    "Mike is so hansome, and he's the greatest teacher!",
    "Honestly I can't beleive he would do that",
    "I think I would like some ice cream",
    "Professional athletes are paid way too much",
    "How much money do you think he makes",
    "I need to hire someone new",
    "There's always so much construction in the city",
    "Traffic is going to be really bad",
    "I'd like to live in NYC!"
        ]

labels = [0]*11+[1]*11+[4]*11

In [ ]:
def average_cosin(v1s, v2s):
    assert len(v1s) == len(v2s)
    avg_sim = []
    for i_1 in v1s:
        for i_2 in v2s:
            avg_sim.append(cosine_sim(i_1, i_2))
    return avg_sim

# Similarity with pos and neg sents
w2v_neg = [nlp(sent).vector for sent in sents[0:11]]
w2v_pos = [nlp(sent).vector for sent in sents[11:22]]

w2v_sims = average_cosin(w2v_neg, w2v_pos)
print("MEAN SIMILARITY:", np.mean(w2v_sims))
plt.scatter(range(len(w2v_sims)), w2v_sims, c='b')
plt.show()

In [ ]:
embs = np.zeros((len(sents), 300))
for i, sent in enumerate(sents):
    embs[i] = nlp(sent).vector
embs.shape

In [ ]:
reduce = Isomap(n_components=3)
sv_reduced = reduce.fit_transform(embs)
sv_reduced.shape

params = dict(size=6,color=labels,colorscale='Viridis')
fig = go.Figure(data=[go.Scatter3d(x=sv_reduced[:,0],
                                   y=sv_reduced[:,1],
                                   z=sv_reduced[:,2],
                                   mode='markers',
                                   marker=params,
                                    hovertext=sents,
                                  )])

fig.update_layout(
    autosize=False,
    width=900,
    height=700,
    margin=dict(l=50, r=50, b=100, t=100, pad=2),
    paper_bgcolor="LightSteelBlue",
)

fig.show()

## Sentence similarity with Google Sentence Encoder

In [ ]:
# conda install conda-forge::tensorflow-hub

In [ ]:
#@title Load the Universal Sentence Encoder's TF Hub module
from absl import logging

import tensorflow as tf
import tensorflow_hub as hub
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import seaborn as sns

module_url = "https://tfhub.dev/google/universal-sentence-encoder/4" #@param ["https://tfhub.dev/google/universal-sentence-encoder/4", "https://tfhub.dev/google/universal-sentence-encoder-large/5"]
model = hub.load(module_url)
print ("module %s loaded" % module_url)
def embed(input):
    return model(input)

In [ ]:
google_embs = np.array(embed(sents))
google_embs.shape

In [ ]:
import tensorflow.keras as k
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input, Conv2D

In [ ]:
def average_cosin(v1s, v2s):
    assert len(v1s) == len(v2s)
    avg_sim = []
    for i_1 in v1s:
        for i_2 in v2s:
            avg_sim.append(cosine_sim(i_1, i_2))
    return avg_sim

google_neg = google_embs[0:11]
google_pos = google_embs[11:22]
average_cosin(google_neg, google_pos)

google_sims = average_cosin(google_neg, google_pos)
print("MEAN SIMILARITY:", np.mean(google_sims))
plt.scatter(range(len(w2v_sims)), w2v_sims, c='b', label='W2V avg vecs')
plt.scatter(range(len(google_sims)), google_sims, c='r', label='Google vecs')
plt.legend()
plt.show()

In [ ]:
google_sims

In [ ]:
reduce = Isomap(n_components=3)
sv_reduced = reduce.fit_transform(google_embs)
sv_reduced.shape

params = dict(size=6,color=labels,colorscale='Viridis')
fig = go.Figure(data=[go.Scatter3d(x=sv_reduced[:,0],
                                   y=sv_reduced[:,1],
                                   z=sv_reduced[:,2],
                                   mode='markers',
                                   marker=params,
                                    hovertext=sents,
                                  )])


fig.update_layout(
    autosize=False,
    width=900,
    height=700,
    margin=dict(l=50, r=50, b=100, t=100, pad=2),
    paper_bgcolor="LightSteelBlue",
)

fig.show()

In [ ]:
cosine_sim(google_embs[0], google_embs[-1])

# Exercise solutions

## 1. to 8.

See appendix A.

## 9.

*Exercise: Load the MNIST dataset (introduced in chapter 3) and split it into a training set and a test set (take the first 60,000 instances for training, and the remaining 10,000 for testing).*

The MNIST dataset was loaded earlier.

In [ ]:
X_train = mnist['data'][:60000]
y_train = mnist['target'][:60000]

X_test = mnist['data'][60000:]
y_test = mnist['target'][60000:]

*Exercise: Train a Random Forest classifier on the dataset and time how long it takes, then evaluate the resulting model on the test set.*

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf = RandomForestClassifier(n_estimators=100, random_state=42)

In [ ]:
import time

t0 = time.time()
rnd_clf.fit(X_train, y_train)
t1 = time.time()

In [ ]:
print("Training took {:.2f}s".format(t1 - t0))

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = rnd_clf.predict(X_test)
accuracy_score(y_test, y_pred)

*Exercise: Next, use PCA to reduce the dataset's dimensionality, with an explained variance ratio of 95%.*

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95)
X_train_reduced = pca.fit_transform(X_train)

*Exercise: Train a new Random Forest classifier on the reduced dataset and see how long it takes. Was training much faster?*

In [ ]:
rnd_clf2 = RandomForestClassifier(n_estimators=100, random_state=42)
t0 = time.time()
rnd_clf2.fit(X_train_reduced, y_train)
t1 = time.time()

In [ ]:
print("Training took {:.2f}s".format(t1 - t0))

Oh no! Training is actually more than twice slower now! How can that be? Well, as we saw in this chapter, dimensionality reduction does not always lead to faster training time: it depends on the dataset, the model and the training algorithm. See figure 8-6 (the `manifold_decision_boundary_plot*` plots above). If you try a softmax classifier instead of a random forest classifier, you will find that training time is reduced by a factor of 3 when using PCA. Actually, we will do this in a second, but first let's check the precision of the new random forest classifier.

*Exercise: Next evaluate the classifier on the test set: how does it compare to the previous classifier?*

In [ ]:
X_test_reduced = pca.transform(X_test)

y_pred = rnd_clf2.predict(X_test_reduced)
accuracy_score(y_test, y_pred)

It is common for performance to drop slightly when reducing dimensionality, because we do lose some useful signal in the process. However, the performance drop is rather severe in this case. So PCA really did not help: it slowed down training and reduced performance. :(

Let's see if it helps when using softmax regression:

In [ ]:
from sklearn.linear_model import LogisticRegression

log_clf = LogisticRegression(multi_class="multinomial", solver="lbfgs", random_state=42, max_iter=1000)
t0 = time.time()
log_clf.fit(X_train, y_train)
t1 = time.time()

In [ ]:
print("Training took {:.2f}s".format(t1 - t0))

In [ ]:
y_pred = log_clf.predict(X_test)
accuracy_score(y_test, y_pred)

Okay, so softmax regression takes much longer to train on this dataset than the random forest classifier, plus it performs worse on the test set. But that's not what we are interested in right now, we want to see how much PCA can help softmax regression. Let's train the softmax regression model using the reduced dataset:

In [ ]:
log_clf2 = LogisticRegression(multi_class="multinomial", solver="lbfgs", random_state=42, max_iter=1000)
t0 = time.time()
log_clf2.fit(X_train_reduced, y_train)
t1 = time.time()

In [ ]:
print("Training took {:.2f}s".format(t1 - t0))

Nice! Reducing dimensionality led to a 4× speedup. :)  Let's check the model's accuracy:

In [ ]:
y_pred = log_clf2.predict(X_test_reduced)
accuracy_score(y_test, y_pred)

A very slight drop in performance, which might be a reasonable price to pay for a 4× speedup, depending on the application.

So there you have it: PCA can give you a formidable speedup... but not always!

## 10.

*Exercise: Use t-SNE to reduce the MNIST dataset down to two dimensions and plot the result using Matplotlib. You can use a scatterplot using 10 different colors to represent each image's target class.*

The MNIST dataset was loaded above.

Dimensionality reduction on the full 60,000 images takes a very long time, so let's only do this on a random subset of 10,000 images:

In [ ]:
np.random.seed(42)

m = 10000
idx = np.random.permutation(60000)[:m]

X = mnist['data'][idx]
y = mnist['target'][idx]

Now let's use t-SNE to reduce dimensionality down to 2D so we can plot the dataset:

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42)
X_reduced = tsne.fit_transform(X)

Now let's use Matplotlib's `scatter()` function to plot a scatterplot, using a different color for each digit:

In [ ]:
plt.figure(figsize=(13,10))
plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=y, cmap="jet")
plt.axis('off')
plt.colorbar()
plt.show()

Isn't this just beautiful? :) This plot tells us which numbers are easily distinguishable from the others (e.g., 0s, 6s, and most 8s are rather well separated clusters), and it also tells us which numbers are often hard to distinguish (e.g., 4s and 9s, 5s and 3s, and so on).

Let's focus on digits 3 and 5, which seem to overlap a lot.

In [ ]:
plt.figure(figsize=(9,9))
cmap = mpl.cm.get_cmap("jet")
for digit in (2, 3, 5):
    plt.scatter(X_reduced[y == digit, 0], X_reduced[y == digit, 1], c=[cmap(digit / 9)])
plt.axis('off')
plt.show()

Let's see if we can produce a nicer image by running t-SNE on these 3 digits:

In [ ]:
idx = (y == 2) | (y == 3) | (y == 5) 
X_subset = X[idx]
y_subset = y[idx]

tsne_subset = TSNE(n_components=2, random_state=42)
X_subset_reduced = tsne_subset.fit_transform(X_subset)

In [ ]:
plt.figure(figsize=(9,9))
for digit in (2, 3, 5):
    plt.scatter(X_subset_reduced[y_subset == digit, 0], X_subset_reduced[y_subset == digit, 1], c=[cmap(digit / 9)])
plt.axis('off')
plt.show()

Much better, now the clusters have far less overlap. But some 3s are all over the place. Plus, there are two distinct clusters of 2s, and also two distinct clusters of 5s. It would be nice if we could visualize a few digits from each cluster, to understand why this is the case. Let's do that now. 

*Exercise: Alternatively, you can write colored digits at the location of each instance, or even plot scaled-down versions of the digit images themselves (if you plot all digits, the visualization will be too cluttered, so you should either draw a random sample or plot an instance only if no other instance has already been plotted at a close distance). You should get a nice visualization with well-separated clusters of digits.*

Let's create a `plot_digits()` function that will draw a scatterplot (similar to the above scatterplots) plus write colored digits, with a minimum distance guaranteed between these digits. If the digit images are provided, they are plotted instead. This implementation was inspired from one of Scikit-Learn's excellent examples ([plot_lle_digits](http://scikit-learn.org/stable/auto_examples/manifold/plot_lle_digits.html), based on a different digit dataset).

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from matplotlib.offsetbox import AnnotationBbox, OffsetImage

def plot_digits(X, y, min_distance=0.05, images=None, figsize=(13, 10)):
    # Let's scale the input features so that they range from 0 to 1
    X_normalized = MinMaxScaler().fit_transform(X)
    # Now we create the list of coordinates of the digits plotted so far.
    # We pretend that one is already plotted far away at the start, to
    # avoid `if` statements in the loop below
    neighbors = np.array([[10., 10.]])
    # The rest should be self-explanatory
    plt.figure(figsize=figsize)
    cmap = mpl.cm.get_cmap("jet")
    digits = np.unique(y)
    for digit in digits:
        plt.scatter(X_normalized[y == digit, 0], X_normalized[y == digit, 1], c=[cmap(digit / 9)])
    plt.axis("off")
    ax = plt.gcf().gca()  # get current axes in current figure
    for index, image_coord in enumerate(X_normalized):
        closest_distance = np.linalg.norm(np.array(neighbors) - image_coord, axis=1).min()
        if closest_distance > min_distance:
            neighbors = np.r_[neighbors, [image_coord]]
            if images is None:
                plt.text(image_coord[0], image_coord[1], str(int(y[index])),
                         color=cmap(y[index] / 9), fontdict={"weight": "bold", "size": 16})
            else:
                image = images[index].reshape(28, 28)
                imagebox = AnnotationBbox(OffsetImage(image, cmap="binary"), image_coord)
                ax.add_artist(imagebox)

Let's try it! First let's just write colored digits:

In [ ]:
plot_digits(X_reduced, y)

Well that's okay, but not that beautiful. Let's try with the digit images:

In [ ]:
plot_digits(X_reduced, y, images=X, figsize=(35, 25))

In [ ]:
plot_digits(X_subset_reduced, y_subset, images=X_subset, figsize=(22, 22))

*Exercise: Try using other dimensionality reduction algorithms such as PCA, LLE, or MDS and compare the resulting visualizations.*

Let's start with PCA. We will also time how long it takes:

In [ ]:
from sklearn.decomposition import PCA
import time

t0 = time.time()
X_pca_reduced = PCA(n_components=2, random_state=42).fit_transform(X)
t1 = time.time()
print("PCA took {:.1f}s.".format(t1 - t0))
plot_digits(X_pca_reduced, y)
plt.show()

Wow, PCA is blazingly fast! But although we do see a few clusters, there's way too much overlap. Let's try LLE:

In [ ]:
from sklearn.manifold import LocallyLinearEmbedding

t0 = time.time()
X_lle_reduced = LocallyLinearEmbedding(n_components=2, random_state=42).fit_transform(X)
t1 = time.time()
print("LLE took {:.1f}s.".format(t1 - t0))
plot_digits(X_lle_reduced, y)
plt.show()

That took a while, and the result does not look too good. Let's see what happens if we apply PCA first, preserving 95% of the variance:

In [ ]:
from sklearn.pipeline import Pipeline

pca_lle = Pipeline([
    ("pca", PCA(n_components=0.95, random_state=42)),
    ("lle", LocallyLinearEmbedding(n_components=2, random_state=42)),
])
t0 = time.time()
X_pca_lle_reduced = pca_lle.fit_transform(X)
t1 = time.time()
print("PCA+LLE took {:.1f}s.".format(t1 - t0))
plot_digits(X_pca_lle_reduced, y)
plt.show()

The result is more or less the same, but this time it was almost 4× faster.

Let's try MDS. It's much too long if we run it on 10,000 instances, so let's just try 2,000 for now:

In [ ]:
from sklearn.manifold import MDS

m = 2000
t0 = time.time()
X_mds_reduced = MDS(n_components=2, random_state=42).fit_transform(X[:m])
t1 = time.time()
print("MDS took {:.1f}s (on just 2,000 MNIST images instead of 10,000).".format(t1 - t0))
plot_digits(X_mds_reduced, y[:m])
plt.show()

Meh. This does not look great, all clusters overlap too much. Let's try with PCA first, perhaps it will be faster?

In [ ]:
from sklearn.pipeline import Pipeline

pca_mds = Pipeline([
    ("pca", PCA(n_components=0.95, random_state=42)),
    ("mds", MDS(n_components=2, random_state=42)),
])
t0 = time.time()
X_pca_mds_reduced = pca_mds.fit_transform(X[:2000])
t1 = time.time()
print("PCA+MDS took {:.1f}s (on 2,000 MNIST images).".format(t1 - t0))
plot_digits(X_pca_mds_reduced, y[:2000])
plt.show()

Same result, and no speedup: PCA did not help (or hurt).

Let's try LDA:

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

t0 = time.time()
X_lda_reduced = LinearDiscriminantAnalysis(n_components=2).fit_transform(X, y)
t1 = time.time()
print("LDA took {:.1f}s.".format(t1 - t0))
plot_digits(X_lda_reduced, y, figsize=(12,12))
plt.show()

This one is very fast, and it looks nice at first, until you realize that several clusters overlap severely.

Well, it's pretty clear that t-SNE won this little competition, wouldn't you agree? We did not time it, so let's do that now:

In [ ]:
from sklearn.manifold import TSNE

t0 = time.time()
X_tsne_reduced = TSNE(n_components=2, random_state=42).fit_transform(X)
t1 = time.time()
print("t-SNE took {:.1f}s.".format(t1 - t0))
plot_digits(X_tsne_reduced, y)
plt.show()

It's twice slower than LLE, but still much faster than MDS, and the result looks great. Let's see if a bit of PCA can speed it up:

In [ ]:
pca_tsne = Pipeline([
    ("pca", PCA(n_components=0.95, random_state=42)),
    ("tsne", TSNE(n_components=2, random_state=42)),
])
t0 = time.time()
X_pca_tsne_reduced = pca_tsne.fit_transform(X)
t1 = time.time()
print("PCA+t-SNE took {:.1f}s.".format(t1 - t0))
plot_digits(X_pca_tsne_reduced, y)
plt.show()

Yes, PCA roughly gave us a 25% speedup, without damaging the result. We have a winner!